# Pipeline con LangChain — resumen → traducción → verificación con gate (Colab opcional · e6)

Este notebook es el **camino "pro" OPCIONAL** del ejercicio estrella e6. Reproduce **el mismo
pipeline** que hiciste a mano (`INSTRUCCIONES.md`), pero **automatizado con LangChain**: cada paso
es una *chain* (`prompt | modelo | parser`) y el **gate** es una función de Python que parsea el JSON
del verificador y reintenta la traducción si hace falta.

**Está pensado para no programadores.** Solo tienes que:
1. Ejecutar las celdas **de arriba hacia abajo** (botón ▶ a la izquierda de cada celda, o `Shift+Enter`).
2. Pegar tu **API key** en la celda indicada.
3. Leer la salida de cada paso y el veredicto del gate.

> No necesitas entender el código: los comentarios en español explican cada parte. Todo va envuelto
> en `try/except`, así que si algo falla (sin key, sin crédito, error de red) verás el error en la
> celda en vez de que el notebook se rompa.

> **No es necesario para aprobar e6.** El camino principal es el encadenamiento **manual** del chat.
> Esto es para quien quiera automatizar y medir a escala.

## Paso 1 — Instalar LangChain y el proveedor

Instalamos `langchain` (el framework de encadenamiento) y `langchain-openai` (el conector a los
modelos de OpenAI). En Colab tarda ~30 segundos. Si ves un aviso de "reiniciar entorno", puedes
ignorarlo.

In [ ]:
# Instala LangChain y el conector de OpenAI (silencioso con -q).
!pip install -q langchain langchain-openai
print("Listo. Continua con el Paso 2.")

## Paso 2 — Pega aquí tu API key

Usamos un modelo de **OpenAI**. Obtén tu clave en https://platform.openai.com/api-keys y **pégala
entre las comillas**.

> ⚠️ **No compartas esta clave ni subas el notebook con la clave pegada.** Es como una contraseña:
> da acceso a tu cuenta y puede generar gastos. Si la expones por error, bórrala (revoke) en el panel
> del proveedor y crea una nueva.

In [ ]:
# Pega tu clave entre las comillas (debe empezar con "sk-").
OPENAI_API_KEY = ""   # ej: "sk-..."

import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

if not OPENAI_API_KEY:
    print("⚠️  No pegaste la API key. Vuelve a esta celda, pega la clave y ejecutala de nuevo.")
else:
    print("Clave cargada.")

## Paso 3 — Crear el modelo

Creamos el objeto del modelo que usarán los tres pasos. `temperature=0` lo hace estable (mismo input,
misma salida), ideal para un pipeline.

> **Nombres de modelos:** los proveedores los cambian seguido. Si el nombre quedara obsoleto,
> reemplaza el valor de `MODELO` por el modelo disponible que veas en el panel de OpenAI.

In [ ]:
from langchain_openai import ChatOpenAI

MODELO = "gpt-4o-mini"  # cambialo si OpenAI renombro sus modelos (ver tu panel)

try:
    modelo = ChatOpenAI(model=MODELO, temperature=0)
    print(f"Modelo '{MODELO}' listo.")
except Exception as e:
    print(f"[ERROR al crear el modelo] {e}")

## Paso 4 — El texto fuente

Es el mismo texto de `texto-fuente.md` (el único input externo del pipeline). Puedes reemplazarlo por
cualquier texto de ~1 pagina pegandolo entre las triples comillas.

In [ ]:
TEXTO_FUENTE = """\
Durante los primeros anios de adopcion de los modelos de lenguaje, el patron dominante fue el
"mega-prompt": una sola instruccion que le pedia al modelo resolver una tarea compleja de una sola
pasada. Un equipo de soporte escribia un unico prompt que debia clasificar el correo de un cliente,
traducirlo, redactar una respuesta y verificar que cumpliera la politica de la empresa.

El problema aparecio con la escala: cuando estos mega-prompts fallaban, fallaban de forma opaca, y
nadie podia decir en que paso se habia torcido el resultado. Como todo ocurria en una sola llamada,
el sistema era una caja negra.

La alternativa fue descomponer la tarea en una secuencia de pasos pequenios, donde cada paso hace una
sola cosa y procesa la salida del anterior. Anthropic, en su guia de 2024, llamo a este patron
"encadenamiento de prompts" y senialo que intercambia algo de latencia por mayor precision, porque
cada llamada enfrenta una tarea mas facil. Entre paso y paso se inserta un "gate": una regla que
evalua la salida y decide si el proceso continua, se rehace o se escala a una persona, deteniendo la
propagacion de errores."""

print("Texto fuente cargado (", len(TEXTO_FUENTE), "caracteres).")

## Paso 5 — Definir los 3 pasos como *chains*

Cada paso es una **chain** de LangChain compuesta con LCEL (el operador `|`), que **enlaza** un
prompt, el modelo y un parser: `prompt | modelo | parser`. Son los mismos 3 prompts del pipeline
manual (`paso1-resumen.md`, `paso2-traduccion.md`, `paso3-verificacion.md`).

- **Paso 1 (resumen):** devuelve texto plano.
- **Paso 2 (traduccion):** recibe el resumen, devuelve texto plano en ingles.
- **Paso 3 (verificacion):** recibe resumen + traduccion, devuelve **JSON** (el gate).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- Paso 1: Resumen (texto plano) ---
prompt_resumen = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente que resume textos de forma concisa y factual."),
    ("human",
     "Resume el siguiente texto en MAXIMO 3 oraciones, sin opiniones ni informacion que no este "
     "en el texto. Conserva el idioma original (espaniol). Devuelve SOLO el resumen.\n\n"
     "<<<\n{texto}\n>>>"),
])
chain_resumen = prompt_resumen | modelo | StrOutputParser()

# --- Paso 2: Traduccion (texto plano en ingles) ---
# Acepta una 'instruccion_extra' opcional que usaremos en el reintento del gate (revise).
prompt_traduccion = ChatPromptTemplate.from_messages([
    ("system", "Eres un traductor profesional."),
    ("human",
     "Traduce al INGLES el siguiente RESUMEN, manteniendo precision, significado y tono formal. "
     "No agregues ni omitas informacion. {instruccion_extra} Devuelve SOLO la traduccion.\n\n"
     "RESUMEN:\n<<<\n{resumen}\n>>>"),
])
chain_traduccion = prompt_traduccion | modelo | StrOutputParser()

# --- Paso 3: Verificacion / gate (JSON) ---
prompt_verificacion = ChatPromptTemplate.from_messages([
    ("system", "Eres un verificador de consistencia bilingue."),
    ("human",
     "Compara el RESUMEN (espaniol) con su TRADUCCION (ingles) y determina si la traduccion es FIEL "
     "(no omite informacion ni cambia el significado). Devuelve SOLO un objeto JSON valido, sin texto "
     "fuera del JSON, con EXACTAMENTE estas claves: "
     '{{"faithful": boolean, "missing_info": [string], "changes_of_meaning": [string], '
     '"action": "approve" | "revise"}}. Usa action=approve si faithful=true, si no action=revise.\n\n'
     "RESUMEN:\n<<<\n{resumen}\n>>>\n\nTRADUCCION:\n<<<\n{traduccion}\n>>>"),
])
chain_verificacion = prompt_verificacion | modelo | StrOutputParser()

print("Las 3 chains estan definidas (prompt | modelo | parser).")

## Paso 6 — El gate en Python

El gate es una funcion que **parsea el JSON** del verificador y decide la accion. Es el mismo gate
del pipeline manual, ahora en codigo:

```
if action == "approve":                          -> termina OK
if action == "revise" and retries <  max_retries: -> rehacer la traduccion
if action == "revise" and retries >= max_retries: -> escalate (a un humano)
```

El modelo a veces envuelve el JSON en ```json ... ```; la funcion lo limpia antes de parsear, y si
aun asi no parsea, devuelve `escalate` (no podemos confiar en una salida ilegible).

In [ ]:
import json, re

def parsear_json(texto):
    """Extrae y parsea el JSON del verificador. Tolera ```json ... ``` alrededor."""
    limpio = texto.strip()
    # Quita vallas de codigo si las hay.
    limpio = re.sub(r"^```(?:json)?", "", limpio).strip()
    limpio = re.sub(r"```$", "", limpio).strip()
    # Si quedara texto alrededor, intenta tomar el primer objeto {...}.
    inicio, fin = limpio.find("{"), limpio.rfind("}")
    if inicio != -1 and fin != -1:
        limpio = limpio[inicio:fin + 1]
    return json.loads(limpio)


def aplicar_gate(salida_verificador, retries, max_retries=2):
    """Decide approve | revise | escalate a partir del JSON del verificador.
    Devuelve (decision, datos) donde datos es el JSON parseado (o el error)."""
    try:
        datos = parsear_json(salida_verificador)
    except Exception as e:
        # Si el JSON no parsea, no podemos confiar en la salida -> escalamos.
        return "escalate", {"error": f"JSON invalido: {e}", "raw": salida_verificador}

    action = datos.get("action", "revise")
    if action == "approve":
        return "approve", datos
    if action == "revise" and retries < max_retries:
        return "revise", datos
    return "escalate", datos  # revise pero ya sin reintentos disponibles

print("Gate listo.")

## Paso 7 — Correr el pipeline completo (con el gate y los reintentos)

Esta funcion encadena los 3 pasos **igual que a mano**: corre el resumen, pasa su salida a la
traduccion, pasa ambas a la verificacion, aplica el gate y, si dice `revise`, **rehace la traduccion**
con las correcciones del verificador, hasta aprobar o agotar los reintentos (`escalate`).

Imprime la salida de **cada paso** para que veas el encadenamiento.

In [ ]:
def correr_pipeline(texto, max_retries=2):
    try:
        # --- Paso 1: Resumen ---
        resumen = chain_resumen.invoke({"texto": texto})
        print("=== PASO 1 · RESUMEN ===\n" + resumen + "\n")

        instruccion_extra = ""   # vacio en el primer intento
        retries = 0

        while True:
            # --- Paso 2: Traduccion (consume la salida del Paso 1) ---
            traduccion = chain_traduccion.invoke({
                "resumen": resumen,
                "instruccion_extra": instruccion_extra,
            })
            print(f"=== PASO 2 · TRADUCCION (intento {retries + 1}) ===\n" + traduccion + "\n")

            # --- Paso 3: Verificacion (consume resumen + traduccion) ---
            verif_raw = chain_verificacion.invoke({
                "resumen": resumen,
                "traduccion": traduccion,
            })
            print("=== PASO 3 · VERIFICACION (JSON del gate) ===\n" + verif_raw + "\n")

            # --- Gate ---
            decision, datos = aplicar_gate(verif_raw, retries, max_retries)
            print(f"--> GATE: {decision.upper()}  (reintentos usados: {retries})\n")

            if decision == "approve":
                print("✅ Pipeline APROBADO. Traduccion final:\n" + traduccion)
                return {"resultado": "approve", "traduccion": traduccion, "retries": retries}

            if decision == "escalate":
                print("🚩 ESCALATE: se agotaron los reintentos o el JSON no parseo. Requiere revisor humano.")
                return {"resultado": "escalate", "traduccion": traduccion, "retries": retries, "datos": datos}

            # decision == revise: preparamos la instruccion de correccion y reintentamos.
            faltantes = datos.get("missing_info", []) + datos.get("changes_of_meaning", [])
            instruccion_extra = "Corrige especificamente esto: " + "; ".join(faltantes) + "."
            retries += 1

    except Exception as e:
        print(f"[ERROR en el pipeline] {e}")
        return {"resultado": "error", "error": str(e)}


# Corre el pipeline sobre el texto fuente.
resultado = correr_pipeline(TEXTO_FUENTE)

## Paso 8 (opcional) — Medir sobre un mini-dataset (observabilidad)

Para reproducir la **observabilidad** de la clase: corre el pipeline sobre **varios textos** y calcula
el **revise rate** (cuantas corridas necesitaron al menos un `revise`) y el **escalate rate**. Asi se
ve cual es el paso mas debil **con numeros**, no a ojo.

> Sugerencia: agrega 4-5 textos cortos a la lista. Aqui va uno solo de ejemplo para que la celda corra.

In [ ]:
mini_dataset = [
    TEXTO_FUENTE,
    "Las energias renovables crecieron de forma sostenida en la ultima decada. La solar y la eolica "
    "bajaron de costo hasta competir con los combustibles fosiles. El reto pendiente es almacenar la "
    "energia para cuando no hay sol ni viento.",
    # Agrega aqui mas textos para medir mejor.
]

resumen_metricas = {"approve": 0, "escalate": 0, "con_revise": 0, "total": 0, "error": 0}

for i, t in enumerate(mini_dataset, start=1):
    print(f"\n########## TEXTO {i} de {len(mini_dataset)} ##########\n")
    r = correr_pipeline(t)
    resumen_metricas["total"] += 1
    estado = r.get("resultado", "error")
    if estado in resumen_metricas:
        resumen_metricas[estado] += 1
    if r.get("retries", 0) > 0:
        resumen_metricas["con_revise"] += 1

total = max(resumen_metricas["total"], 1)  # evita division por cero
print("\n=== METRICAS DEL PIPELINE ===")
print(f"Corridas: {resumen_metricas['total']}")
print(f"Revise rate (paso traduccion): {resumen_metricas['con_revise'] / total:.0%}")
print(f"Escalate rate: {resumen_metricas['escalate'] / total:.0%}")
print("El paso mas debil suele ser la TRADUCCION (mayor revise rate), no el resumen.")

## Que acabas de hacer

Reprodujiste el pipeline manual de e6 en codigo: **3 chains encadenadas** (la salida de una es la
entrada de la siguiente) + un **gate** que parsea el JSON del verificador y reintenta o escala. Es
exactamente el patron de *prompt chaining* de Anthropic (2024), con el **gate** que detiene la
propagacion de errores y el **HITL** (`escalate`) cuando se agotan los reintentos.

La unica diferencia con hacerlo a mano es **quien copia y pega**: aqui lo hace el codigo. Si en vez de
un orden fijo dejaras que el **modelo decidiera** que paso correr a continuacion, dejaria de ser un
*workflow* y pasaria a ser un *agent* (Anthropic 2024).